# Database Schema Documentation and Sample Data Showcase

This notebook demonstrates the database schemas for PostgreSQL and Qdrant, inserts sample data, and shows how to query both databases.


## PostgreSQL Schema

The PostgreSQL database contains two main tables:

### 1. `logs` Table
Stores raw log entries from various services.

**Schema:**
- `id` (SERIAL PRIMARY KEY): Auto-incrementing unique identifier
- `timestamp` (TIMESTAMPTZ): When the log was created (defaults to NOW())
- `level` (TEXT): Log level (e.g., 'error', 'warning', 'info', 'debug')
- `raw` (JSONB): Raw log data as JSON, can contain any fields like:
  - `message`: The log message
  - `service` or `service_name`: Service identifier
  - `traceback`: Stack traces for errors
  - Any other service-specific fields

### 2. `reports` Table
Stores analysis reports generated by the agent system.

**Schema:**
- `id` (SERIAL PRIMARY KEY): Auto-incrementing unique identifier
- `created_at` (TIMESTAMPTZ): When the report was created (defaults to NOW())
- `level` (TEXT): Log level of the analyzed log
- `service` (TEXT): Service name
- `content` (TEXT): The analysis report content
- `raw_log` (TEXT): The original log that was analyzed

**Indexes:**
- `idx_reports_created_at`: Index on `created_at` for efficient time-based queries


## Qdrant Schema

The Qdrant vector database stores error-fix pairs for RAG (Retrieval-Augmented Generation) operations.

### Collection: `log_fixes`

**Configuration:**
- **Collection Name**: `log_fixes` (configurable via `QDRANT_COLLECTION` env var)
- **Vector Dimension**: 1536 (for `text-embedding-3-small` model)
- **Distance Metric**: COSINE (for similarity search)
- **Sparse Vectors**: BM25 configured with IDF modifier for hybrid search

**Content Structure:**
Each document in the collection contains:
- **Vector Embedding**: 1536-dimensional vector generated from error-fix text pairs
- **Payload/Content**: Combined text in format:
  ```
  Error: <error_log_message>
  
  Fix: <fix_description>
  ```
- **Metadata** (optional): Additional information like:
  - Service name
  - Severity level
  - Date resolved
  - Any other relevant context

**Usage:**
The vector database enables semantic search to find similar past errors and their solutions, even when the exact error message doesn't match.


In [1]:
# Import required libraries
import os
import json
from datetime import datetime, timedelta
from agent_system.tools.db_tool import query_logs, get_logs_by_error_pattern
from agent_system.tools.rag_tool import search_fixes_for_error, add_fix_to_knowledge_base
from agent_system.core.storage import insert_log, insert_report
from dotenv import load_dotenv

load_dotenv()

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


## Insert Sample Data into PostgreSQL

Let's insert sample log entries into the `logs` table:


In [5]:
# Sample log entries to insert
sample_logs = [
    {
        "level": "error",
        "raw": {
            "message": "Bank gateway timeout after 5000ms",
            "service": "payment-service",
            "timestamp": "2025-01-14T09:42:17.883Z",
            "traceback": "Traceback (most recent call last):\n  File \"/app/payment/processor.py\", line 214, in process_payment\n    result = bank_api.charge(card, amount)\n  File \"/app/common/bank_api.py\", line 87, in charge\n    raise ConnectionError(\"Bank gateway timeout after 5000ms\")\nConnectionError: Bank gateway timeout after 5000ms"
        }
    },
    {
        "level": "error",
        "raw": {
            "message": "Database connection pool exhausted",
            "service": "user-service",
            "timestamp": "2025-01-14T10:15:32.123Z",
            "error_code": "DB_POOL_EXHAUSTED",
            "details": "Max connections: 100, Active: 100, Waiting: 5"
        }
    },
    {
        "level": "warning",
        "raw": {
            "message": "High memory usage detected",
            "service": "analytics-service",
            "timestamp": "2025-01-14T11:20:45.567Z",
            "memory_usage_percent": 85,
            "threshold": 80
        }
    },
    {
        "level": "error",
        "raw": {
            "message": "Failed to authenticate user",
            "service": "auth-service",
            "timestamp": "2025-01-14T12:05:10.234Z",
            "user_id": "user_12345",
            "reason": "Invalid API key"
        }
    },
    {
        "level": "info",
        "raw": {
            "message": "Service health check passed",
            "service": "payment-service",
            "timestamp": "2025-01-14T13:30:00.000Z",
            "status": "healthy",
            "response_time_ms": 45
        }
    }
]

# Insert logs
inserted_logs = []
for log_entry in sample_logs:
    try:
        result = insert_log(level=log_entry["level"], raw=log_entry["raw"])
        inserted_logs.append(result)
        print(f"✅ Inserted log ID {result['id']}: {log_entry['raw']['message'][:50]}...")
    except Exception as e:
        print(f"❌ Failed to insert log: {e}")

print(f"\n📊 Total logs inserted: {len(inserted_logs)}")


✅ Inserted log ID 9: Bank gateway timeout after 5000ms...
✅ Inserted log ID 10: Database connection pool exhausted...
✅ Inserted log ID 11: High memory usage detected...
✅ Inserted log ID 12: Failed to authenticate user...
✅ Inserted log ID 13: Service health check passed...

📊 Total logs inserted: 5


In [6]:
# Insert sample reports
sample_reports = [
    {
        "level": "error",
        "service": "payment-service",
        "content": "Analysis: Bank gateway timeout error detected. This is a critical issue affecting payment processing. Recommended actions: 1) Increase timeout threshold, 2) Implement retry logic with exponential backoff, 3) Add circuit breaker pattern.",
        "raw_log": "Bank gateway timeout after 5000ms"
    },
    {
        "level": "error",
        "service": "user-service",
        "content": "Analysis: Database connection pool exhausted. This indicates high load or connection leaks. Recommended actions: 1) Review connection pool settings, 2) Check for connection leaks, 3) Consider scaling database resources.",
        "raw_log": "Database connection pool exhausted"
    }
]

# Insert reports
inserted_reports = []
for report in sample_reports:
    try:
        result = insert_report(
            level=report["level"],
            service=report["service"],
            content=report["content"],
            raw_log=report["raw_log"]
        )
        inserted_reports.append(result)
        print(f"✅ Inserted report ID {result['id']} for service: {result['service']}")
    except Exception as e:
        print(f"❌ Failed to insert report: {e}")

print(f"\n📊 Total reports inserted: {len(inserted_reports)}")


✅ Inserted report ID 5 for service: payment-service
✅ Inserted report ID 6 for service: user-service

📊 Total reports inserted: 2


## Insert Sample Data into Qdrant

Now let's add sample error-fix pairs to the Qdrant vector database:


In [10]:
# Sample error-fix pairs to add to Qdrant (with concrete, actionable fixes)
error_fix_pairs = [
    {
        "error": "Bank gateway timeout after 5000ms",
        "fix": "Fix in file: /app/common/bank_api.py, line 87\n\n1. Increase timeout in bank_api.py line 45:\n   Change: TIMEOUT = 5000\n   To: TIMEOUT = 10000\n\n2. Add retry logic in /app/payment/processor.py, line 214:\n   Replace:\n   result = bank_api.charge(card, amount)\n   \n   With:\n   from tenacity import retry, stop_after_attempt, wait_exponential\n   \n   @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=4))\n   def charge_with_retry(card, amount):\n       return bank_api.charge(card, amount)\n   \n   result = charge_with_retry(card, amount)\n\n3. Add circuit breaker in /app/common/circuit_breaker.py (new file):\n   Implement circuit breaker pattern to prevent cascading failures.",
        "metadata": {
            "service": "payment-service",
            "severity": "high",
            "resolved_date": "2025-01-10",
            "category": "timeout",
            "files": ["/app/common/bank_api.py", "/app/payment/processor.py"]
        }
    },
    {
        "error": "Database connection pool exhausted",
        "fix": "Fix in file: /app/config/database.py, line 23\n\n1. Increase pool size in database.py line 23:\n   Change: pool_size=100\n   To: pool_size=150\n\n2. Fix connection leak in /app/services/user_service.py, line 156:\n   Replace:\n   conn = db.get_connection()\n   cursor = conn.cursor()\n   # ... query code ...\n   \n   With:\n   conn = db.get_connection()\n   try:\n       cursor = conn.cursor()\n       # ... query code ...\n   finally:\n       cursor.close()\n       conn.close()\n\n3. Add monitoring in /app/monitoring/pool_monitor.py:\n   Alert when pool usage exceeds 80%.",
        "metadata": {
            "service": "user-service",
            "severity": "critical",
            "resolved_date": "2025-01-08",
            "category": "database",
            "files": ["/app/config/database.py", "/app/services/user_service.py"]
        }
    },
    {
        "error": "Failed to authenticate user: Invalid API key",
        "fix": "Fix in file: /app/auth/validator.py, line 89\n\n1. Add key expiration check in validator.py line 89:\n   Add before validation:\n   if api_key.expires_at and api_key.expires_at < datetime.now():\n       raise AuthenticationError('API key expired')\n\n2. Fix key format validation in /app/auth/validator.py, line 95:\n   Change: if not api_key.startswith('sk_'):\n   To: if not re.match(r'^sk_[a-zA-Z0-9]{32}$', api_key):\n\n3. Add rotation mechanism in /app/auth/key_rotation.py:\n   Implement automatic key rotation every 90 days.\n\n4. Add logging in /app/auth/validator.py, line 102:\n   logger.warning(f'Authentication failed for key: {api_key[:8]}...', extra={'user_id': user_id})",
        "metadata": {
            "service": "auth-service",
            "severity": "medium",
            "resolved_date": "2025-01-12",
            "category": "authentication",
            "files": ["/app/auth/validator.py"]
        }
    },
    {
        "error": "Connection refused: Unable to connect to Redis server",
        "fix": "Fix in file: /app/cache/redis_client.py, line 34\n\n1. Fix connection string in .env file:\n   Change: REDIS_URL=redis://localhost:6379\n   To: REDIS_URL=redis://redis.internal:6379\n   (or verify correct host in docker-compose.yml)\n\n2. Add retry logic in redis_client.py line 34:\n   Replace:\n   self.client = redis.Redis.from_url(redis_url)\n   \n   With:\n   from tenacity import retry, stop_after_attempt, wait_exponential\n   \n   @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=3))\n   def connect_redis(self, redis_url):\n       return redis.Redis.from_url(redis_url, socket_connect_timeout=5)\n   \n   self.client = connect_redis(redis_url)\n\n3. Add fallback in /app/cache/cache_manager.py, line 67:\n   if not redis_available:\n       return local_cache.get(key)",
        "metadata": {
            "service": "cache-service",
            "severity": "high",
            "resolved_date": "2025-01-09",
            "category": "connection",
            "files": ["/app/cache/redis_client.py", ".env", "docker-compose.yml"]
        }
    },
    {
        "error": "Out of memory: Java heap space",
        "fix": "Fix in file: /app/analytics/config/jvm.conf\n\n1. Increase heap size in jvm.conf:\n   Change: JAVA_OPTS=-Xmx2g\n   To: JAVA_OPTS=-Xmx4g -XX:+UseG1GC\n\n2. Fix memory leak in /app/analytics/DataProcessor.java, line 234:\n   Replace:\n   List<Data> results = new ArrayList<>();\n   // ... processing ...\n   return results;\n   \n   With:\n   try (Stream<Data> stream = dataSource.stream()) {\n       return stream\n           .limit(10000)  // Add pagination\n           .collect(Collectors.toList());\n   }\n\n3. Add pagination in /app/analytics/DataProcessor.java, line 189:\n   Process data in batches of 1000 records instead of loading all at once.",
        "metadata": {
            "service": "analytics-service",
            "severity": "critical",
            "resolved_date": "2025-01-11",
            "category": "memory",
            "files": ["/app/analytics/config/jvm.conf", "/app/analytics/DataProcessor.java"]
        }
    }
]

# Add error-fix pairs to Qdrant
added_fixes = []
for pair in error_fix_pairs:
    try:
        result = add_fix_to_knowledge_base(
            error_log=pair["error"],
            fix_description=pair["fix"],
            metadata=pair["metadata"]
        )
        if result.get("success"):
            added_fixes.append(pair)
            print(f"✅ Added fix for: {pair['error'][:50]}...")
        else:
            print(f"❌ Failed to add fix: {result.get('error', 'Unknown error')}")
    except Exception as e:
        print(f"❌ Exception adding fix: {e}")

print(f"\n📊 Total error-fix pairs added: {len(added_fixes)}")


❌ Failed to add fix: Vector database not configured
❌ Failed to add fix: Vector database not configured


KeyboardInterrupt: 

### Add a Single Fix to Qdrant

Example: Adding a specific error-fix pair for "Bank gateway timeout after 5000ms"


In [11]:
# Add a single error-fix pair for "Bank gateway timeout after 5000ms"
bank_timeout_fix = {
    "error": "Bank gateway timeout after 5000ms",
    "fix": "Fix in file: /app/common/bank_api.py, line 87\n\n1. Increase timeout in bank_api.py line 45:\n   Change: TIMEOUT = 5000\n   To: TIMEOUT = 10000\n\n2. Add retry logic in /app/payment/processor.py, line 214:\n   Replace:\n   result = bank_api.charge(card, amount)\n   \n   With:\n   from tenacity import retry, stop_after_attempt, wait_exponential\n   \n   @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=4))\n   def charge_with_retry(card, amount):\n       return bank_api.charge(card, amount)\n   \n   result = charge_with_retry(card, amount)\n\n3. Add circuit breaker in /app/common/circuit_breaker.py (new file):\n   Implement circuit breaker pattern to prevent cascading failures.",
    "metadata": {
        "service": "payment-service",
        "severity": "high",
        "resolved_date": "2025-01-10",
        "category": "timeout",
        "files": ["/app/common/bank_api.py", "/app/payment/processor.py"]
    }
}

# Add the fix to the knowledge base
result = add_fix_to_knowledge_base(
    error_log=bank_timeout_fix["error"],
    fix_description=bank_timeout_fix["fix"],
    metadata=bank_timeout_fix["metadata"]
)

if result.get("success"):
    print(f"✅ Successfully added fix for: {bank_timeout_fix['error']}")
    print(f"   Service: {bank_timeout_fix['metadata']['service']}")
    print(f"   Severity: {bank_timeout_fix['metadata']['severity']}")
    print(f"   Category: {bank_timeout_fix['metadata']['category']}")
else:
    print(f"❌ Failed to add fix: {result.get('error', 'Unknown error')}")


❌ Failed to add fix: Vector database not configured


### Query PostgreSQL: Get logs by level


In [12]:
# Query all error-level logs
error_logs = query_logs(level="error", limit=10)
print(f"Found {len(error_logs)} error logs:\n")
for log in error_logs:
    print(f"ID: {log['id']}, Timestamp: {log['timestamp']}")
    print(f"  Message: {log['raw'].get('message', 'N/A')}")
    print(f"  Service: {log['raw'].get('service', 'N/A')}")
    print()


Found 7 error logs:

ID: 12, Timestamp: 2025-11-18 15:43:19.453153+00:00
  Message: Failed to authenticate user
  Service: auth-service

ID: 10, Timestamp: 2025-11-18 15:43:19.441337+00:00
  Message: Database connection pool exhausted
  Service: user-service

ID: 9, Timestamp: 2025-11-18 15:43:19.419574+00:00
  Message: Bank gateway timeout after 5000ms
  Service: payment-service

ID: 1, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Bank gateway timeout after 5000ms
  Service: payment-service

ID: 6, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Connection refused: Unable to connect to Redis server
  Service: cache-service

ID: 2, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Database connection pool exhausted
  Service: user-service

ID: 4, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Failed to authenticate user
  Service: auth-service



### Query PostgreSQL: Get logs by service


In [ ]:
# Query logs for a specific service
payment_logs = query_logs(service="payment-service", limit=10)
print(f"Found {len(payment_logs)} logs for payment-service:\n")
for log in payment_logs:
    print(f"ID: {log['id']}, Level: {log['level']}, Timestamp: {log['timestamp']}")
    print(f"  Message: {log['raw'].get('message', 'N/A')}")
    print()


Found 4 logs for payment-service:

ID: 13, Level: info, Timestamp: 2025-11-18 15:43:19.458259+00:00
  Message: Service health check passed

ID: 9, Level: error, Timestamp: 2025-11-18 15:43:19.419574+00:00
  Message: Bank gateway timeout after 5000ms

ID: 1, Level: error, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Bank gateway timeout after 5000ms

ID: 5, Level: info, Timestamp: 2025-11-17 21:04:13.239885+00:00
  Message: Service health check passed



### Query PostgreSQL: Search logs by error pattern


In [ ]:
# Search for logs containing "timeout"
timeout_logs = get_logs_by_error_pattern("timeout", limit=10)
print(f"Found {len(timeout_logs)} logs containing 'timeout':\n")
for log in timeout_logs:
    print(f"ID: {log['id']}, Level: {log['level']}")
    print(f"  Message: {log['raw'].get('message', 'N/A')}")
    print(f"  Service: {log['raw'].get('service', 'N/A')}")
    print()


Found 2 logs containing 'timeout':

ID: 9, Level: error
  Message: Bank gateway timeout after 5000ms
  Service: payment-service

ID: 1, Level: error
  Message: Bank gateway timeout after 5000ms
  Service: payment-service



### Query Qdrant: Search for similar errors using RAG


In [2]:
# Search for fixes related to a timeout error
from agent_system.tools.rag_tool import search_fixes_for_error

query_error = "Connection timeout when calling external API"
results = search_fixes_for_error(query_error, top_k=3)

print(f"Searching for fixes related to: '{query_error}'\n")
print(f"Found {len(results)} similar fixes:\n")

for i, result in enumerate(results, 1):
    if "error" in result:
        print(f"❌ {result['error']}")
        if "suggestion" in result:
            print(f"   Suggestion: {result['suggestion']}")
    elif "message" in result:
        print(f"ℹ️  {result['message']}")
        if "suggestion" in result:
            print(f"   Suggestion: {result['suggestion']}")
    else:
        print(f"Result {i}:")
        print(f"  Score: {result.get('score', 'N/A'):.4f}")
        print(f"  Content: {result.get('content', 'N/A')[:200]}...")
        if result.get('metadata'):
            print(f"  Metadata: {result['metadata']}")
    print()


Searching for fixes related to: 'Connection timeout when calling external API'

Found 1 similar fixes:

❌ RAG search failed: 'VectorRetriever' object has no attribute 'retrieve'
   Suggestion: Check vector database connection and configuration



In [1]:
# Search for fixes related to a database error
from agent_system.tools.rag_tool import search_fixes_for_error
from dotenv import load_dotenv

load_dotenv()

query_error2 = "Database connection failed"
results2 = search_fixes_for_error(query_error2, top_k=3)

print(f"Searching for fixes related to: '{query_error2}'\n")
print(f"Found {len(results2)} similar fixes:\n")

for i, result in enumerate(results2, 1):
    if "error" in result:
        print(f"❌ {result['error']}")
        if "suggestion" in result:
            print(f"   Suggestion: {result['suggestion']}")
    elif "message" in result:
        print(f"ℹ️  {result['message']}")
        if "suggestion" in result:
            print(f"   Suggestion: {result['suggestion']}")
    else:
        print(f"Result {i}:")
        print(f"  Score: {result.get('score', 'N/A'):.4f}")
        print(f"  Content: {result.get('content', 'N/A')[:200]}...")
        if result.get('metadata'):
            print(f"  Metadata: {result['metadata']}")
    print()


Searching for fixes related to: 'Database connection failed'

Found 1 similar fixes:

Result 1:
  Score: 0.0000
  Content: {'text': 'No suitable information retrieved from  with similarity_threshold = 0.7.'}...



In [17]:
print(os.getenv("QDRANT_URL"))

https://7ac382c0-35d0-498d-9bb3-66f519d57d62.eu-central-1-0.aws.cloud.qdrant.io:6333
